# Decision Tree Classification (CART) — Google Colab

**Goal:** Predict a **binary class** (0/1) using **Decision Tree Classification (CART)** — readable **if/else rules** that split data to maximize class purity.

| Example | Feature(s) (X) | Target (y) | Dataset |
|---------|----------------|------------|---------|
| **Example 1** | Age, EstimatedSalary | Purchased (0/1) | `Datasets/social_network_ads.csv` |
| **Example 2** | CreditScore, Income, LoanAmount, YearsEmployed | Approved (0/1) | `Datasets/loan_approval.csv` |

| Phase | Topic | Cells |
|-------|-------|-------|
| Phase 0 | Setup | Install & Imports |
| — | Algorithm Guide | Gini impurity, tree structure, CART |
| Phase 1 | Data Pre-processing | Load → Cleaning → Encoding → Split |
| Phase 2 | Algorithm | Train → Predict → Visualize → Evaluate |

> **Run:** Runtime → Run all (or Ctrl+F9)

---
# Algorithm Guide — Decision Tree Classification (CART)

## What is CART for Classification?

**CART** = **Classification And Regression Trees**.  
For classification, the tree splits data to **maximize class purity** and predicts the **majority class** in each leaf.

| Part | Name | Role |
|------|------|------|
| **Root** | Root node | First split — top of the tree |
| **Internal node** | Branch | Decision: `feature ≤ threshold?` |
| **Leaf** | Terminal node | Final prediction = **majority class** in that leaf |

## How Splitting Works (Gini Impurity)

At each node, CART tries every feature and threshold to find the split that **reduces Gini impurity** the most.

**Gini impurity** (2 classes):

`Gini = 1 − Σ pᵢ²`

where **pᵢ** = proportion of class **i** in the node.

| Gini | Meaning |
|------|---------|
| **0** | Pure node — all samples same class |
| **0.5** | Maximum impurity — 50/50 mix (binary) |

**Split score:** choose the split with the **largest impurity reduction** (greedy algorithm).

## Alternative Criterion: Entropy

`Entropy = − Σ pᵢ · log₂(pᵢ)`

sklearn parameter: `criterion='gini'` (default) or `'entropy'` (Information Gain).

## Prediction Rule

1. Start at the **root** node.
2. Follow branches: `if X ≤ threshold` → left, else → right.
3. When you reach a **leaf**, predict the **majority class** of training samples in that leaf.
4. `predict_proba()` = class fractions in the leaf.

## Key Hyperparameters

| Parameter | Role | Effect |
|-----------|------|--------|
| `max_depth` | Max tree depth | Shallow = simpler, Deep = overfitting |
| `min_samples_split` | Min samples to split a node | Higher = simpler tree |
| `min_samples_leaf` | Min samples in a leaf | Higher = smoother decisions |
| `max_leaf_nodes` | Max number of leaves | Limits tree complexity |
| `criterion` | `'gini'` or `'entropy'` | Split quality measure |
| `random_state` | Random seed | Reproducible tree structure |

## Decision Tree vs Other Classifiers

| | Logistic Regression | K-NN | Decision Tree (CART) |
|---|---------------------|------|----------------------|
| Boundary | Linear | Local / flexible | **Axis-aligned**, step regions |
| Scaling | Recommended | Required | **Not required** |
| Interpretability | Coefficients | Low | **High** (`plot_tree`) |
| Overfitting | Low (simple) | Depends on K | **High** if too deep |

## What the Student Must Remember

1. CART uses **Gini** (or Entropy) to choose splits in classification.
2. Leaf prediction = **majority class** in that region.
3. **No feature scaling** needed for Decision Trees.
4. Control **`max_depth`** to avoid overfitting.
5. Use **`plot_tree`** to read exact if/else rules.

## Phase 0 — Cell 0: Install Libraries

Google Colab usually includes most libraries. This cell ensures required packages are available.

**What this cell does:** Installs scikit-learn, pandas, matplotlib, numpy, and seaborn quietly.

In [ ]:
# Install required libraries quietly (-q hides output)
!pip install -q scikit-learn pandas matplotlib numpy seaborn

## Phase 0 — Cell 1: Import Libraries

Import libraries for preprocessing, Decision Tree classification, and evaluation.

**What this cell does:** Loads numpy, pandas, matplotlib, sklearn DecisionTreeClassifier, plot_tree, and metrics.

In [ ]:
# --- Import libraries ---
import numpy as np              # Numerical operations and arrays
import pandas as pd             # Load and manipulate tabular data
import matplotlib.pyplot as plt # Create charts and plots
import seaborn as sns           # Statistical visualizations (heatmaps)

from sklearn.model_selection import train_test_split       # Split data into train/test
from sklearn.impute import SimpleImputer                   # Fill missing values
from sklearn.tree import DecisionTreeClassifier, plot_tree # CART classifier and tree diagram
from sklearn.metrics import (                              # Classification metrics
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report
)

plt.rcParams['figure.figsize'] = (10, 6)  # Default plot size: width=10, height=6 inches
sns.set_theme(style='whitegrid')            # Clean white background with grid lines
np.random.seed(42)                          # Fix random seed for reproducible splits

print('Libraries ready')                      # Confirm all imports loaded successfully

---
# Example 1: Social Network Ads — Decision Tree Classification

Predict whether a user **Purchased** from **Age** and **EstimatedSalary** using CART.

| Column | Role | Description |
|--------|------|-------------|
| `Age` | Feature (X₁) | User age in years |
| `EstimatedSalary` | Feature (X₂) | Estimated annual salary in USD |
| `Purchased` | Target (y) | 0 = No, 1 = Yes |

**File:** `Datasets/social_network_ads.csv`

---
# Phase 1: Data Pre-processing

Prepare the data before training — same template is reused for other algorithms.

## Example 1 — Cell 1: Load and Explore Data

Load the CSV file and perform initial exploration (head, info, describe, shape).

**What this cell does:** Reads `Datasets/social_network_ads.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/social_network_ads.csv')  # Read CSV into a DataFrame

FEATURE_COLS = ['Age', 'EstimatedSalary']  # Two numeric input features
TARGET_COL = 'Purchased'                    # Binary target: 0 = No, 1 = Yes

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nClass balance ({TARGET_COL}):')
print(dataset[TARGET_COL].value_counts())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 1 — Cell 2: Data Cleaning (Handling Missing Values)

Check missing values, remove duplicates, and apply imputation if needed.

**What this cell does:** Cleans the dataset before modeling.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
rows_after = len(dataset)
print(f'\nDuplicates removed: {rows_before - rows_after}')

num_cols = dataset.select_dtypes(include=[np.number]).columns.tolist()
imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
if dataset.isnull().sum().sum() > 0:
    dataset[num_cols] = imputer.fit_transform(dataset[num_cols])
    print('Missing values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {rows_after}')

## Example 1 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')
    print(f'Target column: {TARGET_COL} (already 0/1)')

## Example 1 — Cell 4: Splitting the Data

Define X and y, then split 80/20 with **stratify** to preserve class ratio.

**What this cell does:** Creates feature/target arrays and applies stratified train_test_split.

In [ ]:
# Step 4) Train-Test split (stratified for classification)

X = dataset[FEATURE_COLS].values  # Feature matrix: Age and EstimatedSalary
y = dataset[TARGET_COL].values    # Target vector: Purchased (0 or 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train class counts: {np.bincount(y_train)}')
print(f'y_test class counts:  {np.bincount(y_test)}')

> **Note:** Decision Tree Classification does **not** require feature scaling. Trees split on thresholds — scale does not matter.

---
# Phase 2: Decision Tree Classification — Social Network Ads

Train CART with controlled depth and visualize the tree.

## Example 1 — Cell 5: Train the Model

Train `DecisionTreeClassifier` with `max_depth=5` and `criterion='gini'`.

**What this cell does:** Fits the CART model and prints tree depth and leaf count.

In [ ]:
# Step 5) Train Decision Tree Classifier (CART)

classifier = DecisionTreeClassifier(
    criterion='gini',        # Split by Gini impurity reduction
    max_depth=5,             # Limit depth to prevent overfitting
    min_samples_leaf=5,      # Each leaf must have at least 5 samples
    random_state=42          # Reproducible tree structure
)

classifier.fit(X_train, y_train)  # Build tree: find best Gini splits on training data

print('Decision Tree (CART) trained successfully.')
print(f'Tree depth: {classifier.get_depth()}')           # Actual depth of the built tree
print(f'Number of leaves: {classifier.get_n_leaves()}')  # Total leaf nodes (decision regions)

## Example 1 — Cell 6: Predict

Predict class labels and probabilities on the test set.

**What this cell does:** Routes each sample through the tree to a leaf.

In [ ]:
# Step 6) Predict classes and probabilities

y_pred_train = classifier.predict(X_train)            # Class labels for training set
y_pred_test = classifier.predict(X_test)              # Class labels for test set
y_proba_test = classifier.predict_proba(X_test)[:, 1]  # P(Purchased=1) = leaf class fraction

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    label = 'Yes' if y_pred_test[i] == 1 else 'No'
    actual = 'Yes' if y_test[i] == 1 else 'No'
    print(f'  Actual={actual}, Predicted={label}, P(Purchase)={y_proba_test[i]:.2%}')

## Example 1 — Cell 7: Visualization

Plot the **decision regions** and the **tree diagram** (`plot_tree`).

**What this cell does:** Shows axis-aligned decision boundaries and readable if/else rules.

In [ ]:
# Step 7) Visualization — decision regions + tree diagram

fig, axes = plt.subplots(1, 2, figsize=(16, 6))  # Two subplots: regions and tree

# --- Left: decision regions ---
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 5000, X[:, 1].max() + 5000
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = np.c_[xx.ravel(), yy.ravel()]
Z = classifier.predict(grid).reshape(xx.shape)

axes[0].contourf(xx, yy, Z, alpha=0.25, cmap='RdYlGn')
axes[0].scatter(X_train[y_train == 0, 0], X_train[y_train == 0, 1], c='red', label='Not Purchased', alpha=0.5, s=40)
axes[0].scatter(X_train[y_train == 1, 0], X_train[y_train == 1, 1], c='green', label='Purchased', alpha=0.5, s=40)
axes[0].set_xlabel('Age')
axes[0].set_ylabel('Estimated Salary (USD)')
axes[0].set_title('CART Decision Regions — Social Network Ads')
axes[0].legend(fontsize=8)

# --- Right: tree diagram ---
plot_tree(
    classifier,                         # Trained CART model
    feature_names=FEATURE_COLS,         # Names on split nodes
    class_names=['No', 'Yes'],          # Leaf class labels
    filled=True,                        # Color nodes by majority class
    rounded=True,                       # Rounded node boxes
    fontsize=8,                         # Readable font size
    ax=axes[1]                          # Draw on second subplot
)
axes[1].set_title('CART Tree Structure')

plt.tight_layout()
plt.show()

## Example 1 — Cell 8: Evaluation

Evaluate with Accuracy, Precision, Recall, F1, and Confusion Matrix.

**What this cell does:** Computes classification metrics on the test set.

In [ ]:
# Step 8) Evaluation — classification metrics

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, zero_division=0)
rec = recall_score(y_test, y_pred_test, zero_division=0)
f1 = f1_score(y_test, y_pred_test, zero_division=0)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [acc, prec, rec, f1],
    'Description': [
        'Fraction of correct predictions',
        'Of predicted Yes, fraction actually Yes',
        'Of actual Yes, fraction correctly predicted',
        'Balance between Precision and Recall'
    ]
})

display(results.round(4))

cm = confusion_matrix(y_test, y_pred_test)
print('\nConfusion Matrix (rows=Actual, cols=Predicted):')
print(cm)

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No (0)', 'Yes (1)']).plot(ax=ax, cmap='Blues')
plt.title('Confusion Matrix — Decision Tree (CART)')
plt.tight_layout()
plt.show()

print(f'\nExample 1 Test Accuracy = {acc:.4f}')

## Why does CART work for Social Network Ads?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Non-linear regions** | Purchase behavior splits by Age and Salary thresholds |
| 2 | **No scaling needed** | Trees compare `Age ≤ 35?` — raw values work |
| 3 | **Readable rules** | `plot_tree` shows exact if/else logic for students |
| 4 | **Axis-aligned splits** | Each split is parallel to an axis — easy to interpret |
| 5 | **Control overfitting** | `max_depth` and `min_samples_leaf` limit tree complexity |

> **Summary:** CART creates **rectangular regions** — e.g. "Age > 40 AND Salary > 80k → Purchased".

## Understanding Classification Metrics — Example 1

| Metric | Decision Tree Context |
|--------|----------------------|
| **Accuracy** | Overall correct leaf assignments |
| **Precision** | When tree predicts Yes, how often correct |
| **Recall** | Of all actual Yes, how many the tree catches |
| **F1** | Balance when classes are imbalanced |

> **Tip:** If train accuracy >> test accuracy, the tree is **overfitting** — reduce `max_depth`.

---
# Example 2: Loan Approval — Decision Tree Classification

Predict **Approved** vs **Rejected** from four financial features using CART.

| Column | Role | Description |
|--------|------|-------------|
| `CreditScore` | Feature (X₁) | Credit score (300–850) |
| `Income` | Feature (X₂) | Annual income in USD |
| `LoanAmount` | Feature (X₃) | Requested loan amount in USD |
| `YearsEmployed` | Feature (X₄) | Years at current job |
| `Approved` | Target (y) | 1 = Approved, 0 = Rejected |

**File:** `Datasets/loan_approval.csv`

## Example 2 — Cell 1: Load and Explore Data

Load the loan approval CSV and inspect the data.

**What this cell does:** Reads `Datasets/loan_approval.csv` and displays basic statistics.

In [ ]:
# Step 1) Load dataset
dataset = pd.read_csv('Datasets/loan_approval.csv')

FEATURE_COLS = ['CreditScore', 'Income', 'LoanAmount', 'YearsEmployed']
TARGET_COL = 'Approved'

print('First 5 rows:')
display(dataset.head())

print('\nDataset info:')
dataset.info()

print('\nStatistical summary:')
display(dataset.describe())

print(f'\nClass balance ({TARGET_COL}):')
print(dataset[TARGET_COL].value_counts())

print(f'\nShape: {dataset.shape[0]} rows x {dataset.shape[1]} columns')

## Example 2 — Cell 2: Data Cleaning (Handling Missing Values)

This dataset includes missing values in features to demonstrate `SimpleImputer`.

**What this cell does:** Checks for nulls, removes duplicates, and imputes missing values.

In [ ]:
# Step 2) Data cleaning

print('Missing values per column:')
print(dataset.isnull().sum())

rows_before = len(dataset)
dataset = dataset.drop_duplicates().reset_index(drop=True)
print(f'\nDuplicates removed: {rows_before - len(dataset)}')

if dataset[TARGET_COL].isnull().sum() > 0:
    dataset = dataset.dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print('Rows with missing target removed')

imputer = SimpleImputer(missing_values=np.nan, strategy='mean')
cols = FEATURE_COLS + [TARGET_COL]
if dataset[cols].isnull().sum().sum() > 0:
    dataset[cols] = imputer.fit_transform(dataset[cols])
    print('Missing feature values imputed with mean')
else:
    print('No missing values — imputer not applied')

print(f'\nRows after cleaning: {len(dataset)}')

## Example 2 — Cell 3: Categorical Data Encoding

All columns are numeric — encoding is skipped.

**What this cell does:** Confirms no categorical encoding is needed.

In [ ]:
# Step 3) Categorical encoding

cat_cols = dataset.select_dtypes(include=['object', 'category']).columns.tolist()
if cat_cols:
    print(f'Categorical columns found: {cat_cols}')
else:
    print('No categorical columns — encoding skipped.')
    print(f'Feature columns: {FEATURE_COLS}')

## Example 2 — Cell 4: Splitting the Data

Define X (4 features) and y (Approved), then split 80/20 with stratify.

**What this cell does:** Creates feature/target arrays and applies stratified train_test_split.

In [ ]:
# Step 4) Train-Test split

X = dataset[FEATURE_COLS].values
y = dataset[TARGET_COL].values.astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'X_train shape: {X_train.shape}')
print(f'X_test shape:  {X_test.shape}')
print(f'y_train class counts: {np.bincount(y_train)}')
print(f'y_test class counts:  {np.bincount(y_test)}')

## Example 2 — Cell 5: Train the Model

Train CART with multiple features — tree splits on Credit, Income, Loan, or Years at each node.

**What this cell does:** Fits the model and reports feature importances.

In [ ]:
# Step 5) Train Decision Tree Classifier (CART)

classifier = DecisionTreeClassifier(
    criterion='gini',
    max_depth=5,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42
)

classifier.fit(X_train, y_train)

print('Decision Tree (CART) trained successfully.')
print(f'Tree depth: {classifier.get_depth()}')
print(f'Number of leaves: {classifier.get_n_leaves()}')

print('\nFeature importances:')
for name, imp in zip(FEATURE_COLS, classifier.feature_importances_):
    print(f'  {name:15s} -> {imp:.4f}')

## Example 2 — Cell 6: Predict

Predict loan approval on the test set.

**What this cell does:** Generates predictions by routing samples through the tree.

In [ ]:
# Step 6) Predict

y_pred_test = classifier.predict(X_test)
y_proba_test = classifier.predict_proba(X_test)[:, 1]

print('Sample predictions (Test set):')
for i in range(min(5, len(y_test))):
    actual = 'Approved' if y_test[i] == 1 else 'Rejected'
    pred = 'Approved' if y_pred_test[i] == 1 else 'Rejected'
    print(f'  Actual={actual}, Predicted={pred}, P(Approved)={y_proba_test[i]:.2%}')

## Example 2 — Cell 7: Visualization

Plot **feature importances** and **confusion matrix**.

**What this cell does:** Shows which features matter most for approval decisions.

In [ ]:
# Step 7) Visualization — feature importances + confusion matrix

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(FEATURE_COLS, classifier.feature_importances_, color='teal')
axes[0].set_xlabel('Importance')
axes[0].set_title('Feature Importances — Loan Approval (CART)')

cm = confusion_matrix(y_test, y_pred_test)
ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Rejected', 'Approved']).plot(ax=axes[1], cmap='Blues')
axes[1].set_title('Confusion Matrix — Decision Tree (CART)')

plt.tight_layout()
plt.show()

## Example 2 — Cell 8: Evaluation

Full classification report with Accuracy, Precision, Recall, and F1.

**What this cell does:** Computes and displays evaluation metrics.

In [ ]:
# Step 8) Evaluation

acc = accuracy_score(y_test, y_pred_test)
prec = precision_score(y_test, y_pred_test, zero_division=0)
rec = recall_score(y_test, y_pred_test, zero_division=0)
f1 = f1_score(y_test, y_pred_test, zero_division=0)

results = pd.DataFrame({
    'Metric': ['Accuracy', 'Precision', 'Recall', 'F1 Score'],
    'Value': [acc, prec, rec, f1],
    'Description': [
        'Fraction of correct predictions',
        'Of predicted Approved, fraction actually approved',
        'Of actual Approved, fraction correctly predicted',
        'Balance between Precision and Recall'
    ]
})

display(results.round(4))

print('\nDetailed classification report:')
print(classification_report(y_test, y_pred_test, target_names=['Rejected (0)', 'Approved (1)']))

print(f'\nExample 2 Test Accuracy = {acc:.4f}')

## Why does CART work well for Loan Approval?

| # | Reason | Explanation |
|---|--------|-------------|
| 1 | **Multiple features** | CART splits on Credit, Income, Loan, and Years automatically |
| 2 | **Non-linear rules** | e.g. CreditScore > 620 AND LoanAmount < 100k → Approved |
| 3 | **Feature importances** | Shows which feature contributes most to splits |
| 4 | **No scaling needed** | Raw credit score and income work directly |
| 5 | **High accuracy possible** | With tuned `max_depth`, CART fits complex approval patterns |

> **Compare:** CART vs Logistic Regression — trees capture **non-linear** rules without manual feature engineering.

## Understanding Classification Metrics — Example 2

| Scenario | Focus Metric |
|----------|--------------|
| **False approval costly** | **Precision** |
| **False rejection costly** | **Recall** |
| **Balanced cost** | **Accuracy** or **F1** |

> **Overfitting check:** Compare `accuracy_score(y_train, y_pred_train)` vs test accuracy — large gap means reduce `max_depth`.